In [0]:

from pyspark.sql import functions as F
from delta.tables import DeltaTable

def process_silver(microBatchDF, batch_id):

    if microBatchDF.isEmpty():
        return

    spark = microBatchDF.sparkSession

    # 1. DataFrame recibido desde Bronze
    df_bronze = microBatchDF

    print(f"Processing batch: {batch_id}")

    # 2. Eliminar duplicados
    print("Before deduplication:", df_bronze.count())

    df_deduplicated = df_bronze.dropDuplicates()

    print("After deduplication:", df_deduplicated.count())

    # 3. Comprobar nulos
    print("Nulls by column:")

    df_deduplicated.select([
        F.sum(F.col(c).isNull().cast("int")).alias(c)
        for c in df_deduplicated.columns
    ]).show()

    # 4. Eliminar registros sin end_station_id
    df_nulls_removed = df_deduplicated.dropna(
        subset=["end_station_id"]
    )

    print("Nulls after removal:")

    df_nulls_removed.select([
        F.sum(F.col(c).isNull().cast("int")).alias(c)
        for c in df_nulls_removed.columns
    ]).show()

    # 5. Transformación de tipos
    df_typed = (
        df_nulls_removed
        .withColumn(
            "started_at",
            F.to_timestamp(
                "started_at",
                "yyyy-MM-dd HH:mm:ss.SSS"
            )
        )
        .withColumn(
            "ended_at",
            F.to_timestamp(
                "ended_at",
                "yyyy-MM-dd HH:mm:ss.SSS"
            )
        )
        .withColumn("start_lat", F.col("start_lat").cast("double"))
        .withColumn("start_lng", F.col("start_lng").cast("double"))
        .withColumn("end_lat", F.col("end_lat").cast("double"))
        .withColumn("end_lng", F.col("end_lng").cast("double"))
    )

    # 6. Eliminar duplicados por clave de negocio
    # Una fila por ride_id dentro del microbatch
    df_typed = df_typed.dropDuplicates(["ride_id"])

    # 7. MERGE en Silver
    target = DeltaTable.forName(
        spark,
        "citibike.02_silver.citibikes"
    )

    (
        target.alias("target")
        .merge(
            df_typed.alias("source"),
            "target.ride_id = source.ride_id"
        )
        .whenMatchedUpdateAll(
            condition="""
                source.ingestion_at > target.ingestion_at
            """
        )
        .whenNotMatchedInsertAll()
        .execute()
    )

    print(f"Batch {batch_id} successfully processed")


In [0]:
checkpoint_path = (
    "/Volumes/citibike/01_bronze/checkpoints/bronze_to_silver"
)

query = (
    spark.readStream
    .table("citibike.01_bronze.citibikes_raw")
    .writeStream
    .foreachBatch(process_silver)
    .option(
        "checkpointLocation",
        checkpoint_path
    )
    .trigger(availableNow=True)
    .start()
)

query.awaitTermination()